# 02 - Silver Transformation

## Purpose

Convert raw Bronze tables into clean, typed, deduplicated, and conformed Silver Delta tables.

## Concepts covered

- Column name standardization
- Data type casting
- Text, status, and category cleanup
- Duplicate removal using business keys
- Referential integrity validation
- Quarantine tables for invalid records
- Row count and quality logging

## Prerequisites

01_bronze_ingestion.ipynb completed successfully and Bronze tables exist in the Lakehouse.

## Expected output

Silver tables for customers, accounts, products, branches, and transactions. Optional quarantine tables are created only when invalid records exist.

In [ ]:
from datetime import datetime, timezone
import re
from typing import Iterable, List

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ENTITIES = ["customers", "accounts", "products", "branches", "transactions"]

def log_info(message: str) -> None:
    print(f"[INFO] {datetime.now(timezone.utc).isoformat()} | {message}")

def log_warning(message: str) -> None:
    print(f"[WARN] {datetime.now(timezone.utc).isoformat()} | {message}")

def normalize_column_name(column_name: str) -> str:
    cleaned = column_name.strip().lower()
    cleaned = re.sub(r"[^a-z0-9_]+", "_", cleaned)
    cleaned = re.sub(r"_+", "_", cleaned).strip("_")
    return cleaned

def clean_column_names(df: DataFrame) -> DataFrame:
    result = df
    for old_name in df.columns:
        new_name = normalize_column_name(old_name)
        if old_name != new_name:
            result = result.withColumnRenamed(old_name, new_name)
    return result

def trim_upper(column_name: str):
    return F.upper(F.trim(F.col(column_name)))

def trim_initcap(column_name: str):
    return F.initcap(F.trim(F.col(column_name)))

def trim_lower(column_name: str):
    return F.lower(F.trim(F.col(column_name)))

def parse_bool(column_name: str):
    normalized = F.lower(F.trim(F.col(column_name).cast("string")))
    return F.when(normalized.isin("true", "t", "yes", "y", "1"), F.lit(True)).otherwise(F.lit(False))

def latest_by_key(df: DataFrame, key_columns: Iterable[str]) -> DataFrame:
    order_columns = [F.col("_ingestion_timestamp").desc_nulls_last(), F.col("_batch_id").desc_nulls_last()]
    window_spec = Window.partitionBy(*[F.col(c) for c in key_columns]).orderBy(*order_columns)
    return df.withColumn("_row_number", F.row_number().over(window_spec)).filter(F.col("_row_number") == 1).drop("_row_number")

def add_row_hash(df: DataFrame, columns: List[str]) -> DataFrame:
    values = [F.coalesce(F.col(c).cast("string"), F.lit("")) for c in columns]
    return df.withColumn("_row_hash", F.sha2(F.concat_ws("||", *values), 256))

def write_silver(df: DataFrame, entity_name: str) -> int:
    table_name = f"silver_{entity_name}"
    row_count = df.count()
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    log_info(f"Wrote {table_name} with {row_count} rows")
    return row_count

def write_quarantine(df: DataFrame, table_name: str, reason: str) -> int:
    row_count = df.count()
    if row_count == 0:
        log_info(f"No records quarantined for {table_name}")
        return 0
    enriched = df.withColumn("_quarantine_reason", F.lit(reason)).withColumn("_quarantine_timestamp", F.current_timestamp())
    enriched.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    log_warning(f"Quarantined {row_count} records into {table_name}: {reason}")
    return row_count

def require_table(table_name: str) -> DataFrame:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(f"Required table is missing: {table_name}")
    return spark.table(table_name)

## Load Bronze tables

In [ ]:
bronze = {entity: clean_column_names(require_table(f"bronze_{entity}")) for entity in ENTITIES}

for entity, df in bronze.items():
    log_info(f"bronze_{entity}: {df.count()} rows, {len(df.columns)} columns")

## Transform customers

In [ ]:
customers_silver = (
    bronze["customers"]
    .withColumn("customer_id", trim_upper("customer_id"))
    .withColumn("first_name", trim_initcap("first_name"))
    .withColumn("last_name", trim_initcap("last_name"))
    .withColumn("email", trim_lower("email"))
    .withColumn("phone", F.trim(F.col("phone")))
    .withColumn("city", trim_initcap("city"))
    .withColumn("state", trim_upper("state"))
    .withColumn("country", trim_upper("country"))
    .withColumn("customer_segment", trim_initcap("customer_segment"))
    .withColumn("customer_status", trim_initcap("customer_status"))
    .withColumn("date_of_birth", F.to_date(F.col("date_of_birth"), "yyyy-MM-dd"))
    .withColumn("created_date", F.to_date(F.col("created_date"), "yyyy-MM-dd"))
    .withColumn("is_active_customer", F.col("customer_status") == F.lit("Active"))
)

customers_silver = latest_by_key(customers_silver, ["customer_id"])
customers_silver = add_row_hash(customers_silver, ["customer_id", "first_name", "last_name", "email", "city", "state", "customer_segment", "customer_status"])

invalid_customers = customers_silver.filter(F.col("customer_id").isNull() | (F.trim(F.col("customer_id")) == ""))
write_quarantine(invalid_customers, "quarantine_customers_missing_key", "Missing customer_id")
customers_silver = customers_silver.filter(F.col("customer_id").isNotNull() & (F.trim(F.col("customer_id")) != ""))

write_silver(customers_silver, "customers")
display(customers_silver.orderBy("customer_id"))

## Transform products and branches

In [ ]:
products_silver = (
    bronze["products"]
    .withColumn("product_id", trim_upper("product_id"))
    .withColumn("product_name", trim_initcap("product_name"))
    .withColumn("product_category", trim_initcap("product_category"))
    .withColumn("product_family", trim_initcap("product_family"))
    .withColumn("fee_model", trim_initcap("fee_model"))
    .withColumn("is_active", parse_bool("is_active"))
    .withColumn("launch_date", F.to_date(F.col("launch_date"), "yyyy-MM-dd"))
)
products_silver = latest_by_key(products_silver, ["product_id"])
products_silver = add_row_hash(products_silver, ["product_id", "product_name", "product_category", "product_family", "fee_model"])
write_quarantine(products_silver.filter(F.col("product_id").isNull()), "quarantine_products_missing_key", "Missing product_id")
products_silver = products_silver.filter(F.col("product_id").isNotNull())
write_silver(products_silver, "products")

branches_silver = (
    bronze["branches"]
    .withColumn("branch_id", trim_upper("branch_id"))
    .withColumn("branch_name", trim_initcap("branch_name"))
    .withColumn("city", trim_initcap("city"))
    .withColumn("state", trim_upper("state"))
    .withColumn("region", trim_initcap("region"))
    .withColumn("opened_date", F.to_date(F.col("opened_date"), "yyyy-MM-dd"))
    .withColumn("is_active", parse_bool("is_active"))
)
branches_silver = latest_by_key(branches_silver, ["branch_id"])
branches_silver = add_row_hash(branches_silver, ["branch_id", "branch_name", "city", "state", "region"])
write_quarantine(branches_silver.filter(F.col("branch_id").isNull()), "quarantine_branches_missing_key", "Missing branch_id")
branches_silver = branches_silver.filter(F.col("branch_id").isNotNull())
write_silver(branches_silver, "branches")

display(products_silver.orderBy("product_id"))
display(branches_silver.orderBy("branch_id"))

## Transform accounts with referential checks

In [ ]:
accounts_clean = (
    bronze["accounts"]
    .withColumn("account_id", trim_upper("account_id"))
    .withColumn("customer_id", trim_upper("customer_id"))
    .withColumn("product_id", trim_upper("product_id"))
    .withColumn("branch_id", trim_upper("branch_id"))
    .withColumn("account_type", trim_initcap("account_type"))
    .withColumn("account_status", trim_initcap("account_status"))
    .withColumn("opened_date", F.to_date(F.col("opened_date"), "yyyy-MM-dd"))
    .withColumn("closed_date", F.to_date(F.col("closed_date"), "yyyy-MM-dd"))
    .withColumn("current_balance", F.col("current_balance").cast("decimal(18,2)"))
    .withColumn("is_open_account", F.col("account_status") == F.lit("Open"))
)
accounts_clean = latest_by_key(accounts_clean, ["account_id"])

write_quarantine(accounts_clean.filter(F.col("account_id").isNull()), "quarantine_accounts_missing_key", "Missing account_id")
write_quarantine(accounts_clean.alias("a").join(customers_silver.select("customer_id").alias("c"), "customer_id", "left_anti"), "quarantine_accounts_invalid_customer", "customer_id not found in silver_customers")
write_quarantine(accounts_clean.alias("a").join(products_silver.select("product_id").alias("p"), "product_id", "left_anti"), "quarantine_accounts_invalid_product", "product_id not found in silver_products")
write_quarantine(accounts_clean.alias("a").join(branches_silver.select("branch_id").alias("b"), "branch_id", "left_anti"), "quarantine_accounts_invalid_branch", "branch_id not found in silver_branches")

accounts_silver = (
    accounts_clean
    .filter(F.col("account_id").isNotNull())
    .join(customers_silver.select("customer_id"), "customer_id", "inner")
    .join(products_silver.select("product_id"), "product_id", "inner")
    .join(branches_silver.select("branch_id"), "branch_id", "inner")
)
accounts_silver = add_row_hash(accounts_silver, ["account_id", "customer_id", "product_id", "branch_id", "account_type", "account_status", "current_balance"])
write_silver(accounts_silver, "accounts")
display(accounts_silver.orderBy("account_id"))

## Transform transactions with business enrichment

In [ ]:
transactions_clean = (
    bronze["transactions"]
    .withColumn("transaction_id", trim_upper("transaction_id"))
    .withColumn("account_id", trim_upper("account_id"))
    .withColumn("product_id", trim_upper("product_id"))
    .withColumn("branch_id", trim_upper("branch_id"))
    .withColumn("transaction_timestamp", F.to_timestamp(F.col("transaction_timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("transaction_date", F.to_date(F.col("transaction_timestamp")))
    .withColumn("transaction_type", trim_initcap("transaction_type"))
    .withColumn("transaction_channel", trim_initcap("transaction_channel"))
    .withColumn("transaction_amount", F.col("transaction_amount").cast("decimal(18,2)"))
    .withColumn("currency", trim_upper("currency"))
    .withColumn("transaction_status", trim_initcap("transaction_status"))
    .withColumn("absolute_transaction_amount", F.abs(F.col("transaction_amount")).cast("decimal(18,2)"))
    .withColumn("debit_credit_indicator", F.when(F.col("transaction_amount") < 0, F.lit("Debit")).otherwise(F.lit("Credit")))
    .withColumn("is_posted", F.col("transaction_status") == F.lit("Posted"))
    .withColumn("year_month", F.date_format(F.col("transaction_timestamp"), "yyyy-MM"))
)
transactions_clean = latest_by_key(transactions_clean, ["transaction_id"])

write_quarantine(transactions_clean.filter(F.col("transaction_id").isNull()), "quarantine_transactions_missing_key", "Missing transaction_id")
write_quarantine(transactions_clean.alias("t").join(accounts_silver.select("account_id").alias("a"), "account_id", "left_anti"), "quarantine_transactions_invalid_account", "account_id not found in silver_accounts")
write_quarantine(transactions_clean.alias("t").join(products_silver.select("product_id").alias("p"), "product_id", "left_anti"), "quarantine_transactions_invalid_product", "product_id not found in silver_products")
write_quarantine(transactions_clean.alias("t").join(branches_silver.select("branch_id").alias("b"), "branch_id", "left_anti"), "quarantine_transactions_invalid_branch", "branch_id not found in silver_branches")

transactions_silver = (
    transactions_clean
    .filter(F.col("transaction_id").isNotNull())
    .join(accounts_silver.select("account_id"), "account_id", "inner")
    .join(products_silver.select("product_id"), "product_id", "inner")
    .join(branches_silver.select("branch_id"), "branch_id", "inner")
)
transactions_silver = add_row_hash(transactions_silver, ["transaction_id", "account_id", "product_id", "branch_id", "transaction_timestamp", "transaction_type", "transaction_channel", "transaction_amount", "transaction_status"])
write_silver(transactions_silver, "transactions")
display(transactions_silver.orderBy("transaction_timestamp"))

## Silver validation summary

In [ ]:
validation_rows = []
key_columns = {"customers": "customer_id", "accounts": "account_id", "products": "product_id", "branches": "branch_id", "transactions": "transaction_id"}

for entity in ENTITIES:
    bronze_count = spark.table(f"bronze_{entity}").count()
    silver_count = spark.table(f"silver_{entity}").count()
    key_column = key_columns[entity]
    duplicate_count = spark.table(f"silver_{entity}").groupBy(key_column).count().filter(F.col("count") > 1).count()
    validation_rows.append({
        "entity_name": entity,
        "bronze_row_count": bronze_count,
        "silver_row_count": silver_count,
        "duplicate_business_keys": duplicate_count,
        "status": "PASS" if duplicate_count == 0 and silver_count <= bronze_count else "REVIEW",
    })

silver_validation_df = spark.createDataFrame(validation_rows)
display(silver_validation_df.orderBy("entity_name"))

if silver_validation_df.filter(F.col("duplicate_business_keys") > 0).count() > 0:
    raise RuntimeError("Silver duplicate key validation failed")

## Next steps

Run 03_gold_dimensional_model.ipynb to create the business-ready star schema used by SQL analytics and Power BI.